<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/05b_near_duplicate_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Do the near-duplicates change the results?

Notebook 01b flagged 51 of the 2,992 test images (1.7%) as possible near-duplicates of training or validation images.
Some are real copies, others false matches. If a real copy sits in both training and test, the CNN may have seen that test image
during training, which would make its test results look better than they should.

Instead of rebuilding the splits and retraining, this recomputes every saved result **without all 51 flagged images**, real or not,
and shows how much each number moves. If the changes are small next to the effects the thesis reports, the conclusions don't depend on the leak.
Provenance and the watermark don't learn from training data, so only the CNN (and fusion, through the CNN) can be affected.

It only rereads the saved per-image results from notebooks 05 and 01b: **no GPU, no retraining**. Runtime > Change runtime type > **CPU**. About a minute.

## 1. Get the code

In [1]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e .

/content
Cloning into 'repo'...
remote: Enumerating objects: 414, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 414 (delta 102), reused 67 (delta 39), pack-reused 236 (from 1)
Receiving objects: 100% (414/414), 6.06 MiB | 8.19 MiB/s, done.
Resolving deltas: 100% (202/202), done.
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for attribution (pyproject.toml) ... done


## 2. Link the saved results on Google Drive

In [2]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
for d in ["evaluation", "data_quality"]:
    !mkdir -p $D/$d && rm -rf outputs/$d && ln -s $D/$d outputs/$d
NEAR = "outputs/data_quality/near_duplicates.csv"
RUNS = ["resnet50", "efficientnet_b0", "resnet50_no_watermark", "efficientnet_b0_no_watermark"]
missing = [p for p in [NEAR] + [f"outputs/evaluation/{r}/per_image.csv" for r in RUNS] if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"missing {missing}: run notebook 01b and notebook 05 (sections 4 and 7) first")
print("found the near-duplicate report and all four evaluation runs")

Mounted at /content/drive
found the near-duplicate report and all four evaluation runs


## 3. Recompute without the flagged images

In [3]:
for run in RUNS:
    arch, _, rest = run.partition("_no_")
    flag = "--no-watermark" if rest else ""
    !python scripts/evaluate.py --arch $arch $flag --resummarize --exclude-near-duplicates $NEAR

Recomputed outputs/evaluation/resnet50/summary.csv and outputs/evaluation/resnet50/summary.json from outputs/evaluation/resnet50/per_image.csv (26928 rows)
resnet50: excluded 51 of 2992 test images (1.7%) flagged as near-duplicates
  provenance_only: largest change  accuracy=0.000  f1=0.000  recall=0.000  fpr=0.000
   watermark_only: largest change  accuracy=0.000  f1=0.001  recall=0.001  fpr=0.000
         cnn_only: largest change  accuracy=0.002  f1=0.002  recall=0.002  fpr=0.001
     multi_signal: largest change  accuracy=0.000  f1=0.001  recall=0.001  fpr=0.001
  written: outputs/evaluation/resnet50/near_duplicate_sensitivity.csv, outputs/evaluation/resnet50/summary_excl_near_duplicates.csv
Recomputed outputs/evaluation/efficientnet_b0/summary.csv and outputs/evaluation/efficientnet_b0/summary.json from outputs/evaluation/efficientnet_b0/per_image.csv (26928 rows)
efficientnet_b0: excluded 51 of 2992 test images (1.7%) flagged as near-duplicates
  provenance_only: largest change  a

## 4. How much did the results move?
**change** = result without the flagged images − result with all images. Positive accuracy change means the model did *better* without them,
so the flagged images were not inflating the score. Compare the size of the changes with the effects the thesis reports
(fusion vs CNN alone: 15–30 points; the watermark's effect on the CNN: 10–22 points).

In [4]:
import json, pandas as pd
pd.set_option("display.width", 200)
overview = []
for run in RUNS:
    info = json.load(open(f"outputs/evaluation/{run}/summary_excl_near_duplicates.json"))
    for system, changes in info["max_abs_change"].items():
        overview.append({"run": run, "system": system, "excluded": info["excluded_images"], **{f"max |Δ {k}|": v for k, v in changes.items()}})
print("Largest change in any condition (absolute value):")
display(pd.DataFrame(overview).round(4))

for run in RUNS:
    s = pd.read_csv(f"outputs/evaluation/{run}/near_duplicate_sensitivity.csv")
    order = s.condition.unique()
    for system in ["cnn_only", "multi_signal"]:
        t = s[s.system == system].set_index("condition").loc[order, ["accuracy_all", "accuracy_without", "accuracy_change", "f1_all", "f1_without", "f1_change"]]
        print(f"\n{run} | {system}")
        display(t.round(4))

Largest change in any condition (absolute value):


,run,system,excluded,max |Δ accuracy|,max |Δ f1|,max |Δ recall|,max |Δ fpr|
0,resnet50,provenance_only,51,0.0000,0.0000,0.0000,0.0000
1,resnet50,watermark_only,51,0.0002,0.0012,0.0011,0.0003
2,resnet50,cnn_only,51,0.0022,0.0016,0.0019,0.0011
3,resnet50,multi_signal,51,0.0003,0.0012,0.0012,0.0006
4,efficientnet_b0,provenance_only,51,0.0000,0.0000,0.0000,0.0000
5,efficientnet_b0,watermark_only,51,0.0002,0.0012,0.0011,0.0003
6,efficientnet_b0,cnn_only,51,0.0025,0.0007,0.0008,0.0017
7,efficientnet_b0,multi_signal,51,0.0013,0.0006,0.0007,0.0013
8,resnet50_no_watermark,provenance_only,51,0.0000,0.0000,0.0000,0.0000
9,resnet50_no_watermark,watermark_only,51,NaN,NaN,0.0000,0.0000



resnet50 | cnn_only


,accuracy_all,accuracy_without,accuracy_change,f1_all,f1_without,f1_change
condition,,,,,,
clean,0.8262,0.8252,-0.0010,0.8012,0.8014,0.0002
jpeg_q90,0.8001,0.7997,-0.0004,0.7719,0.7730,0.0011
jpeg_q70,0.8018,0.8018,-0.0000,0.7720,0.7736,0.0016
jpeg_q50,0.7824,0.7820,-0.0004,0.7503,0.7516,0.0014
jpeg_q30,0.7470,0.7463,-0.0006,0.7021,0.7035,0.0014
webp_q80,0.7774,0.7763,-0.0011,0.7315,0.7319,0.0004
resize_0.5,0.6965,0.6943,-0.0022,0.5831,0.5824,-0.0007
resize_0.25,0.6892,0.6875,-0.0016,0.6131,0.6134,0.0002
recompress_x3,0.8011,0.8004,-0.0007,0.7675,0.7683,0.0008



resnet50 | multi_signal


...

,accuracy_all,accuracy_without,accuracy_change,f1_all,f1_without,f1_change
condition,,,,,,
clean,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000
jpeg_q90,0.9270,0.9265,-0.0005,0.8792,0.8801,0.0009
jpeg_q70,0.8754,0.8747,-0.0007,0.7814,0.7819,0.0004
jpeg_q50,0.8279,0.8269,-0.0011,0.7077,0.7089,0.0012
jpeg_q30,0.7797,0.7791,-0.0005,0.6396,0.6417,0.0020
webp_q80,0.9286,0.9278,-0.0008,0.8561,0.8568,0.0007
resize_0.5,0.8818,0.8808,-0.0010,0.7765,0.7764,-0.0001
resize_0.25,0.8170,0.8161,-0.0008,0.6931,0.6934,0.0003
recompress_x3,0.8830,0.8824,-0.0006,0.7885,0.7886,0.0000



efficientnet_b0_no_watermark | cnn_only


,accuracy_all,accuracy_without,accuracy_change,f1_all,f1_without,f1_change
condition,,,,,,
clean,0.9335,0.9323,-0.0012,0.9336,0.9329,-0.0007
jpeg_q90,0.9188,0.9184,-0.0004,0.9190,0.9191,0.0001
jpeg_q70,0.8556,0.8545,-0.0011,0.8455,0.8453,-0.0002
jpeg_q50,0.8239,0.8225,-0.0014,0.8043,0.8041,-0.0003
jpeg_q30,0.7918,0.7909,-0.0009,0.7618,0.7625,0.0007
webp_q80,0.8957,0.8949,-0.0008,0.8905,0.8904,-0.0001
resize_0.5,0.8529,0.8521,-0.0009,0.8323,0.8325,0.0002
resize_0.25,0.6487,0.6460,-0.0027,0.4624,0.4615,-0.0009
recompress_x3,0.8670,0.8660,-0.0009,0.8558,0.8557,-0.0001



efficientnet_b0_no_watermark | multi_signal


,accuracy_all,accuracy_without,accuracy_change,f1_all,f1_without,f1_change
condition,,,,,,
clean,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000
jpeg_q90,0.9464,0.9455,-0.0009,0.9050,0.9054,0.0005
jpeg_q70,0.8931,0.8925,-0.0007,0.8194,0.8197,0.0004
jpeg_q50,0.8528,0.8518,-0.0010,0.7583,0.7590,0.0007
jpeg_q30,0.8288,0.8280,-0.0007,0.7187,0.7201,0.0013
webp_q80,0.9278,0.9273,-0.0005,0.8499,0.8507,0.0008
resize_0.5,0.8876,0.8865,-0.0012,0.7739,0.7745,0.0006
resize_0.25,0.6667,0.6639,-0.0028,0.3440,0.3425,-0.0016
recompress_x3,0.8997,0.8987,-0.0010,0.8235,0.8231,-0.0004


## 5. Save it
File > Save a copy in GitHub, so the results can be reviewed. The new files (`summary_excl_near_duplicates.*`, `near_duplicate_sensitivity.csv`)
are saved next to each run's results on Drive and are collected by notebook 06.